# USGS Earthquake API — Exploration Notebook

This notebook walks through the USGS Earthquake Hazards Program FDSN Event Web Service.

**API base URL:** `https://earthquake.usgs.gov/fdsnws/event/1/`  
**Docs:** https://earthquake.usgs.gov/fdsnws/event/1/

In [2]:
import requests
import pandas as pd
import json

BASE_URL = "https://earthquake.usgs.gov/fdsnws/event/1/query"

## 1. Check how many events exist for a query

Before fetching everything, it's smart to check the count first.
The API has a hard limit of **20,000 rows per request**.

In [3]:
params = {
    "format": "geojson",
    "starttime": "2023-01-01",
    "endtime": "2023-12-31",
    "minmagnitude": 4.0,
}

r = requests.get(BASE_URL, params=params)
r.raise_for_status()
data = r.json()

print("Total matching events:", data["metadata"]["count"])
print("Status:", data["metadata"]["status"])
print("Generated:", data["metadata"]["generated"])

Total matching events: 16190
Status: 200
Generated: 1774300776000


## 2. Fetch a small sample and inspect the raw GeoJSON

In [4]:
params_sample = {
    "format": "geojson",
    "starttime": "2023-01-01",
    "endtime": "2023-03-31",
    "minmagnitude": 4.0,
    "limit": 5,          # just 5 rows to inspect structure
    "orderby": "time-asc",
}

r = requests.get(BASE_URL, params=params_sample)
r.raise_for_status()
sample = r.json()

# Look at one raw feature
print(json.dumps(sample["features"][0], indent=2))

{
  "type": "Feature",
  "properties": {
    "mag": 4.5,
    "place": "23 km ESE of Manay, Philippines",
    "time": 1672537303755,
    "updated": 1678575105040,
    "tz": null,
    "url": "https://earthquake.usgs.gov/earthquakes/eventpage/us7000j3xk",
    "detail": "https://earthquake.usgs.gov/fdsnws/event/1/query?eventid=us7000j3xk&format=geojson",
    "felt": null,
    "cdi": null,
    "mmi": null,
    "alert": null,
    "status": "reviewed",
    "tsunami": 0,
    "sig": 312,
    "net": "us",
    "code": "7000j3xk",
    "ids": ",us7000j3xk,",
    "sources": ",us,",
    "types": ",origin,phase-data,",
    "nst": 32,
    "dmin": 1.152,
    "rms": 0.47,
    "gap": 104,
    "magType": "mb",
    "type": "earthquake",
    "title": "M 4.5 - 23 km ESE of Manay, Philippines"
  },
  "geometry": {
    "type": "Point",
    "coordinates": [
      126.738,
      7.1397,
      79.194
    ]
  },
  "id": "us7000j3xk"
}


## 3. Parse GeoJSON features into a DataFrame

In [5]:
def parse_usgs_geojson(geojson: dict) -> pd.DataFrame:
    rows = []
    for feat in geojson["features"]:
        props = feat["properties"]
        coords = feat["geometry"]["coordinates"]  # [lon, lat, depth]
        rows.append({
            "id":          feat["id"],
            "time_ms":     props["time"],
            "longitude":   coords[0],
            "latitude":    coords[1],
            "depth_km":    coords[2],
            "magnitude":   props["mag"],
            "mag_type":    props["magType"],
            "place":       props["place"],
            "sig":         props["sig"],
            "felt":        props["felt"],
            "cdi":         props["cdi"],    # community intensity
            "mmi":         props["mmi"],    # ShakeMap max intensity
            "alert":       props["alert"],  # PAGER alert level
            "tsunami":     props["tsunami"],
            "type":        props["type"],
            "url":         props["url"],
        })
    df = pd.DataFrame(rows)
    df["time"] = pd.to_datetime(df["time_ms"], unit="ms", utc=True)
    return df


df_sample = parse_usgs_geojson(sample)
df_sample

,id,time_ms,longitude,latitude,depth_km,magnitude,mag_type,place,sig,felt,cdi,mmi,alert,tsunami,type,url,time
0,us7000j3xk,1672537303755,126.7380,7.1397,79.194,4.5,mb,"23 km ESE of Manay, Philippines",312,None,None,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 01:41:43.755000+00:00
1,us7000j1bs,1672542974442,155.2320,-6.7065,35.000,5.4,mww,"51 km SSW of Panguna, Papua New Guinea",449,None,None,4.445,green,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 03:16:14.442000+00:00
2,us7000j3xm,1672546172814,102.7675,-4.7803,63.787,4.3,mb,"99 km SSW of Pagar Alam, Indonesia",284,None,None,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 04:09:32.814000+00:00
3,us7000j3xu,1672548893914,-177.5423,-19.0419,556.990,4.1,mb,Fiji region,259,None,None,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 04:54:53.914000+00:00
4,us7000j3xn,1672549366402,-174.8756,-15.3219,255.470,4.1,mb,Tonga,259,None,None,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 05:02:46.402000+00:00


## 4. Fetch a full quarter and explore the data

In [6]:
params_full = {
    "format": "geojson",
    "starttime": "2023-01-01",
    "endtime": "2023-03-31",
    "minmagnitude": 4.0,
    "limit": 20000,
    "orderby": "time-asc",
}

r = requests.get(BASE_URL, params=params_full)
r.raise_for_status()
df = parse_usgs_geojson(r.json())

print(f"Rows: {len(df)}")
df.head()

Rows: 4115


,id,time_ms,longitude,latitude,depth_km,magnitude,mag_type,place,sig,felt,cdi,mmi,alert,tsunami,type,url,time
0,us7000j3xk,1672537303755,126.7380,7.1397,79.194,4.5,mb,"23 km ESE of Manay, Philippines",312,NaN,NaN,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 01:41:43.755000+00:00
1,us7000j1bs,1672542974442,155.2320,-6.7065,35.000,5.4,mww,"51 km SSW of Panguna, Papua New Guinea",449,NaN,NaN,4.445,green,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 03:16:14.442000+00:00
2,us7000j3xm,1672546172814,102.7675,-4.7803,63.787,4.3,mb,"99 km SSW of Pagar Alam, Indonesia",284,NaN,NaN,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 04:09:32.814000+00:00
3,us7000j3xu,1672548893914,-177.5423,-19.0419,556.990,4.1,mb,Fiji region,259,NaN,NaN,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 04:54:53.914000+00:00
4,us7000j3xn,1672549366402,-174.8756,-15.3219,255.470,4.1,mb,Tonga,259,NaN,NaN,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 05:02:46.402000+00:00


In [7]:
df.dtypes

id                           str
time_ms                    int64
longitude                float64
latitude                 float64
depth_km                 float64
magnitude                float64
mag_type                     str
place                        str
sig                        int64
felt                     float64
cdi                      float64
mmi                      float64
alert                        str
tsunami                    int64
type                         str
url                          str
time         datetime64[ms, UTC]
dtype: object

In [8]:
df[["magnitude", "depth_km", "sig", "felt", "cdi", "mmi"]].describe()

,magnitude,depth_km,sig,felt,cdi,mmi
count,4115.000000,4115.000000,4115.000000,655.000000,655.000000,248.000000
mean,4.490301,85.654803,316.768408,61.067176,3.570076,3.803714
std,0.384156,140.226721,97.608791,733.804471,1.750261,1.973312
min,4.000000,0.000000,246.000000,0.000000,0.000000,0.000000
25%,4.200000,10.000000,271.000000,1.000000,2.200000,2.585500
50%,4.400000,25.042000,298.000000,2.000000,3.400000,3.685500
75%,4.600000,94.006000,326.000000,8.000000,4.450000,4.802750
max,7.800000,654.010000,2910.000000,18250.000000,9.100000,9.537000


In [9]:
# How many have PAGER alert levels?
df["alert"].value_counts(dropna=False)

alert
NaN       3936
green      165
yellow       9
red          4
orange       1
Name: count, dtype: int64

In [10]:
# Magnitude type breakdown
df["mag_type"].value_counts()

mag_type
mb     3582
mww     291
mwr     200
ml       20
md       16
mw        5
mwc       1
Name: count, dtype: int64

In [11]:
# Missingness summary
df.isna().mean().sort_values(ascending=False).to_frame("pct_missing")

,pct_missing
alert,0.956501
mmi,0.939733
felt,0.840826
cdi,0.840826
id,0.000000
url,0.000000
type,0.000000
tsunami,0.000000
sig,0.000000
time_ms,0.000000


## 5. Try bounding box filters (US only)

In [12]:
# Rough bounding box for the contiguous US
params_us = {
    "format": "geojson",
    "starttime": "2023-01-01",
    "endtime": "2023-12-31",
    "minmagnitude": 3.0,
    "minlatitude":  24.0,
    "maxlatitude":  50.0,
    "minlongitude": -125.0,
    "maxlongitude": -66.0,
    "limit": 20000,
    "orderby": "time-asc",
}

r_us = requests.get(BASE_URL, params=params_us)
r_us.raise_for_status()
df_us = parse_usgs_geojson(r_us.json())

print(f"US events: {len(df_us)}")
df_us.head()

US events: 858


,id,time_ms,longitude,latitude,depth_km,magnitude,mag_type,place,sig,felt,cdi,mmi,alert,tsunami,type,url,time
0,nc73827426,1672553971410,-121.200167,36.595833,8.140000,3.14,ml,"9km NW of Pinnacles, CA",161,27.0,3.4,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 06:19:31.410000+00:00
1,nc73827436,1672555750190,-121.206333,36.595000,7.740000,3.90,mw,"9km NW of Pinnacles, CA",346,272.0,4.1,4.369,green,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 06:49:10.190000+00:00
2,nc73827571,1672598104510,-123.971000,40.409000,30.630000,5.35,mw,"15km SE of Rio Dell, CA",1190,1208.0,5.4,6.853,yellow,1,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 18:35:04.510000+00:00
3,tx2023abqj,1672608227177,-104.421101,31.684255,6.131567,3.30,ml,"54 km S of Whites City, New Mexico",168,3.0,2.7,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-01 21:23:47.177000+00:00
4,tx2023acno,1672650404837,-104.413051,31.687066,6.080151,3.20,ml,"54 km S of Whites City, New Mexico",158,NaN,NaN,NaN,NaN,0,earthquake,https://earthquake.usgs.gov/earthquakes/eventp...,2023-01-02 09:06:44.837000+00:00


## 6. Try other output formats

The API supports `csv` format too, which can be easier to work with for large pulls.

In [13]:
import io

params_csv = {
    "format": "csv",
    "starttime": "2023-01-01",
    "endtime": "2023-01-31",
    "minmagnitude": 4.0,
    "orderby": "time-asc",
}

r_csv = requests.get(BASE_URL, params=params_csv)
r_csv.raise_for_status()

df_csv = pd.read_csv(io.StringIO(r_csv.text))
print(df_csv.columns.tolist())
df_csv.head()

['time', 'latitude', 'longitude', 'depth', 'mag', 'magType', 'nst', 'gap', 'dmin', 'rms', 'net', 'id', 'updated', 'place', 'type', 'horizontalError', 'depthError', 'magError', 'magNst', 'status', 'locationSource', 'magSource']


,time,latitude,longitude,depth,mag,magType,nst,gap,dmin,rms,...,updated,place,type,horizontalError,depthError,magError,magNst,status,locationSource,magSource
0,2023-01-01T01:41:43.755Z,7.1397,126.7380,79.194,4.5,mb,32.0,104.0,1.152,0.47,...,2023-03-11T22:51:45.040Z,"23 km ESE of Manay, Philippines",earthquake,5.51,7.445,0.083,43.0,reviewed,us,us
1,2023-01-01T03:16:14.442Z,-6.7065,155.2320,35.000,5.4,mww,130.0,36.0,3.946,0.68,...,2023-03-11T22:51:29.040Z,"51 km SSW of Panguna, Papua New Guinea",earthquake,8.95,1.770,0.098,10.0,reviewed,us,us
2,2023-01-01T04:09:32.814Z,-4.7803,102.7675,63.787,4.3,mb,17.0,187.0,0.457,0.51,...,2023-03-11T22:51:45.040Z,"99 km SSW of Pagar Alam, Indonesia",earthquake,10.25,6.579,0.238,5.0,reviewed,us,us
3,2023-01-01T04:54:53.914Z,-19.0419,-177.5423,556.990,4.1,mb,15.0,87.0,3.051,0.15,...,2023-03-11T22:51:45.040Z,Fiji region,earthquake,12.85,13.028,0.213,6.0,reviewed,us,us
4,2023-01-01T05:02:46.402Z,-15.3219,-174.8756,255.470,4.1,mb,40.0,81.0,3.413,0.32,...,2023-03-11T22:51:45.040Z,Tonga,earthquake,9.84,6.047,0.095,34.0,reviewed,us,us


## 7. Fetch a single event's detail page

Each event has a `detail` URL with richer data (moment tensor, phase data, etc.)

## 8. Notes for the project

**Key fields we'll use:**
- `magnitude` + `depth_km` — core seismic parameters
- `sig` — USGS significance score (0–1000+), combines mag, felt reports, MMI, and PAGER
- `mmi` — maximum ShakeMap intensity (best proxy for surface shaking)
- `felt` — number of felt reports (community impact proxy)
- `place` — contains the US state abbreviation we'll use for merging with FEMA

**Pagination strategy:** The API caps at 20,000 rows. For a 10-year window we'll
split requests into 3-month chunks and concatenate.

**Missingness to watch out for:**
- `felt`, `cdi`, `mmi` are only populated for events that triggered ShakeMap / DYFI
- `alert` (PAGER) is only populated for significant events
- These will be sparse — that's expected and scientifically meaningful